# Movie Recommendation and Rating Prediction System

## Project Overview

This project develops a movie recommendation and rating prediction system using the ALX Movie Recommendation dataset. It explores user rating behaviour and applies collaborative filtering techniques to identify similar movies and estimate how users may rate movies they have not yet rated.

The project implements item-based cosine similarity for movie recommendations and compares three rating prediction approaches: a movie-average baseline, a personalized model incorporating user preferences, and a regularized model using Bayesian smoothing.

## Project Objectives

- Explore and analyze user–movie rating patterns.
- Identify movies with sufficient ratings for meaningful comparisons.
- Build an item-based recommendation function using cosine similarity.
- Develop and evaluate a baseline rating prediction model.
- Improve predictions by incorporating user rating behaviour.
- Apply Bayesian smoothing to reduce the influence of movies with limited ratings.
- Compare model performance using Root Mean Squared Error (RMSE).
- Generate rating predictions for the Kaggle competition.

## Evaluation Metric

Model performance is evaluated using Root Mean Squared Error (RMSE), which measures the difference between actual and predicted ratings. Lower RMSE values indicate better predictive performance.

## Initial Results

| Model | Validation RMSE |
|---|---:|
| Movie-average baseline | 0.9621 |
| Personalized prediction | 0.8897 |
| Regularized prediction | 0.8867 |

These results provide the starting point for further model improvement.

## 1. Data Loading and Initial Exploration

In this section, we import the Pandas library and load the training and test datasets.

Pandas provides tools for reading, manipulating, and analyzing structured data.

The training dataset contains user–movie interactions and their corresponding ratings, while the test dataset contains user–movie pairs for which ratings must be predicted.

We begin by examining the structure, data types, and sample records to understand the datasets before preprocessing and model development.

In [1]:
import pandas as pd
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

### Previewing the Training Dataset

We display the first five rows of the training dataset to understand its structure and identify the available features.

The dataset contains four columns: `userId`, `movieId`, `rating`, and `timestamp`. Each row represents a user's rating of a particular movie at a recorded time.

Understanding these features helps determine which variables are relevant for building the recommendation and rating prediction models.

In [2]:
train_data.head()

,userId,movieId,rating,timestamp
0,5163,57669,4.0,1518349992
1,106343,5,4.5,1206238739
2,146790,5459,5.0,1076215539
3,106362,32296,2.0,1423042565
4,9041,366,3.0,833375837


### Examining Dataset Structure and Data Types

We use the `info()` method to examine the dataset's dimensions, data types, and non-null values.

The training dataset contains 10,000,038 records and four columns. The `userId`, `movieId`, and `timestamp` columns are integers, while `rating` is a floating-point variable.

All four columns contain non-null values, indicating that there are no missing values in these fields. The dataset occupies approximately 305.2 MB of memory.

Given the large number of records, memory-efficient techniques will be important when constructing the recommendation model.

In [3]:
train_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000038 entries, 0 to 10000037
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 305.2 MB


### Exploring Average Movie Ratings

We group the training data by `movieId` and calculate the mean rating for each movie. The results are sorted in descending order to identify movies with the highest average ratings.

The initial results show several movies with perfect average ratings of 5.0. However, these averages do not account for how many users rated each movie.

A movie with only one rating may appear highly rated without providing enough evidence of consistent viewer satisfaction. Therefore, we will examine rating counts alongside average ratings to obtain more reliable comparisons.

In [3]:
popular_movies = train_data.groupby("movieId")["rating"].mean().sort_values(ascending=False)
popular_movies.head(10)

movieId
208765    5.0
208795    5.0
208939    5.0
142679    5.0
194889    5.0
159117    5.0
159127    5.0
159050    5.0
159053    5.0
106115    5.0
Name: rating, dtype: float64

### Evaluating Movie Ratings and Rating Frequency

To obtain a more reliable assessment of movie ratings, we calculate both the average rating and the total number of ratings received by each movie.

We then exclude movies with 50 or fewer ratings to reduce the influence of movies with limited user feedback.

This approach helps identify highly rated movies supported by a larger number of user reviews, rather than relying on average ratings alone.

In [4]:
movie_stats = train_data.groupby("movieId").agg({
    "rating": ["mean", "count"]
})

movie_stats.columns = ["avg_rating", "rating_count"]

movie_stats = movie_stats[movie_stats["rating_count"] >= 50]

movie_stats.sort_values("avg_rating", ascending=False).head(10)

,avg_rating,rating_count
movieId,,
171011,4.517084,439
159817,4.473294,674
171495,4.456897,116
318,4.417624,32831
170705,4.399130,575
142115,4.337963,216
858,4.311355,20960
100044,4.295455,132
179135,4.291667,264


### Filtering Movies for Collaborative Filtering

To prepare the dataset for collaborative filtering, we retain movies that have received at least 50 ratings.

Movies with very few ratings provide limited information about shared user preferences and may produce less reliable similarity estimates.

The filtered dataset, `df_filtered`, contains user–movie interactions involving movies that meet the minimum rating threshold.

This filtering step also reduces the number of movies considered during similarity calculations, helping manage computational and memory requirements.

In [5]:
# Keep only movies that have at least 50 ratings
movie_counts = train_data["movieId"].value_counts()
active_movies = movie_counts[movie_counts >= 50].index

df_filtered = train_data[train_data["movieId"].isin(active_movies)]

## 2. Building the Movie Recommendation System

### Creating a Sparse User–Movie Matrix

Collaborative filtering uses patterns in user interactions to identify relationships between movies.

We use SciPy's Compressed Sparse Row (CSR) matrix to represent user–movie ratings efficiently.

In this matrix:
- Rows represent individual users.
- Columns represent movies.
- Values represent ratings given by users.
- Missing interactions are stored implicitly as zeros.

A sparse matrix is particularly useful because most users have rated only a small proportion of the available movies.

This approach reduces memory requirements compared with creating a dense user–movie matrix.

In [ ]:
from scipy.sparse import csr_matrix

user_ids = df_filtered["userId"].astype("category").cat.codes
movie_ids = df_filtered["movieId"].astype("category").cat.codes
ratings = df_filtered["rating"]

sparse_matrix = csr_matrix(
    (ratings, (user_ids, movie_ids))
)

### Examining the User–Movie Matrix

We examine the dimensions of the sparse matrix to confirm that user ratings have been represented correctly.

The resulting matrix contains 162,540 users and 9,568 movies.

This matrix provides the foundation for calculating similarities between movies based on their rating patterns.

In [7]:
sparse_matrix.shape

(162540, 9568)

### Calculating Movie Similarity Using Cosine Similarity

Cosine similarity measures the similarity between two vectors based on the angle between them.

We transpose the user–movie matrix so that each row represents a movie and each column represents a user.

We then calculate pairwise cosine similarity between movies using their user rating patterns.

The resulting similarity matrix has dimensions of 9,568 × 9,568, allowing us to identify movies with similar rating patterns.

Higher cosine similarity values indicate greater similarity between the corresponding movie vectors.

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

movie_similarity = cosine_similarity(
    sparse_matrix.T
)

In [9]:
movie_similarity.shape

(9568, 9568)

### Developing the Movie Recommendation Function

After calculating movie similarities, we create a lookup that connects the numerical positions in the similarity matrix to their original movie IDs.

We then define a recommendation function that:

1. Accepts a movie ID as input.
2. Identifies the corresponding movie in the similarity matrix.
3. Retrieves similarity scores between the selected movie and other movies.
4. Sorts movies by similarity score.
5. Excludes the input movie and returns the top five recommendations.

This is an item-based collaborative filtering approach because recommendations are generated from similarities in user rating patterns.

In [10]:
movie_lookup = df_filtered["movieId"].astype("category").cat.categories

In [11]:
movie_lookup[:10]

Index([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype='int64')

In [12]:
import numpy as np

def recommend_movies(movie_id, top_n=5):

    try:
        movie_idx = list(movie_lookup).index(movie_id)
    except ValueError:
        return "Movie not found in filtered dataset."

    similarity_scores = movie_similarity[movie_idx]

    similar_indices = np.argsort(similarity_scores)[::-1][1:top_n+1]

    recommended_movie_ids = movie_lookup[similar_indices]

    return recommended_movie_ids.tolist()

In [13]:
recommend_movies(318)

[296, 593, 356, 527, 50]

In [14]:
test_data.head()

,userId,movieId
0,1,2011
1,1,4144
2,1,5767
3,1,6711
4,1,7318


### Splitting the Data for Model Evaluation

We divide the original training dataset into two subsets:

- 80% training data for learning user and movie rating patterns.
- 20% validation data for evaluating prediction performance.

We use a fixed random state of 42 to ensure reproducibility.

The validation dataset allows us to compare models using ratings that were not included when calculating their prediction statistics.

In [18]:

from sklearn.model_selection import train_test_split

train_set, val_set = train_test_split(
    train_data,
    test_size=0.2,
    random_state=42
)

print("Training set:", train_set.shape)
print("Validation set:", val_set.shape)


Training set: (8000030, 4)
Validation set: (2000008, 4)


In [19]:

# Calculate the average rating for each movie
movie_mean = train_set.groupby("movieId")["rating"].mean()

# Calculate the overall average rating
global_mean = train_set["rating"].mean()

# Predict ratings for the validation set
val_predictions = val_set["movieId"].map(movie_mean)

# Handle movies not found in the training set
val_predictions = val_predictions.fillna(global_mean)

# Display the first five predictions
print(val_predictions.head())


5954604    2.808219
1460784    3.215606
6631832    4.049241
5396407    3.852049
8150654    4.006517
Name: movieId, dtype: float64


### Developing the Baseline Rating Prediction Model

We begin with a simple baseline model that predicts a user's rating based on the average rating received by each movie.

For movies that do not appear in the training subset, we use the overall average rating as a fallback.

We evaluate the model using Root Mean Squared Error (RMSE).

The baseline model achieves a validation RMSE of approximately 0.9621.

This score serves as a benchmark for evaluating whether more advanced prediction methods improve performance.

In [20]:

from sklearn.metrics import mean_squared_error
import numpy as np

# Get actual ratings
actual_ratings = val_set["rating"]

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(actual_ratings, val_predictions))

print("Baseline RMSE:", rmse)


Baseline RMSE: 0.9620988813031842


## 3. Rating Prediction and Model Evaluation

### Inspecting the Test Dataset

The Kaggle test dataset contains user–movie pairs for which ratings must be predicted.

Unlike the training dataset, the test dataset does not include the actual ratings.

Our objective is to estimate these missing ratings using information learned from the training data.

We will first establish a baseline model and then evaluate more personalized prediction approaches.

In [21]:

# Calculate each user's average rating
user_mean = train_set.groupby("userId")["rating"].mean()

# Display the first five users and their average ratings
print(user_mean.head())


userId
1    3.920000
2    3.464286
3    3.703980
4    3.210145
5    3.807692
Name: rating, dtype: float64


In [22]:

# Get each validation user's average rating
val_user_mean = val_set["userId"].map(user_mean)

# Get each validation movie's average rating
val_movie_mean = val_set["movieId"].map(movie_mean)

# Handle users or movies missing from the training set
val_user_mean = val_user_mean.fillna(global_mean)
val_movie_mean = val_movie_mean.fillna(global_mean)

# Combine user and movie averages
personalized_predictions = (
    val_user_mean + val_movie_mean - global_mean
)

# Keep predictions within the valid rating range
personalized_predictions = personalized_predictions.clip(0.5, 5.0)

print(personalized_predictions.head())


5954604    2.192558
1460784    3.299378
6631832    3.842271
5396407    4.472426
8150654    4.408532
dtype: float64


## 4. Personalized Rating Prediction

### Incorporating User Rating Behaviour

The baseline model relies only on movie averages and does not account for differences in individual user preferences.

To improve predictions, we calculate the average rating given by each user.

We then combine user averages with movie averages, adjusting for the global average rating.

The prediction formula is:

Predicted Rating = User Average + Movie Average - Global Average

This approach accounts for users who generally give higher or lower ratings than others.

Predictions are restricted to the valid rating range of 0.5 to 5.0.

The personalized model achieves a validation RMSE of approximately 0.8897, improving upon the baseline RMSE of 0.9621.

In [23]:

# Calculate RMSE for the personalized model
personalized_rmse = np.sqrt(
    mean_squared_error(actual_ratings, personalized_predictions)
)

# Compare both models
print("Baseline RMSE:", rmse)
print("Personalized RMSE:", personalized_rmse)
print("Improvement:", rmse - personalized_rmse)


Baseline RMSE: 0.9620988813031842
Personalized RMSE: 0.88970814883535
Improvement: 0.07239073246783423


### Examining Prediction Errors

We compare actual movie ratings with the predictions generated by the personalized model.

The prediction error is calculated as:

Error = Actual Rating - Predicted Rating

Positive errors indicate that the model underestimated a user's rating, while negative errors indicate that it overestimated the rating.

Examining these differences helps us understand individual prediction errors beyond the overall RMSE score.

In [24]:

# Compare actual ratings with predicted ratings
comparison = val_set[["userId", "movieId", "rating"]].copy()

comparison["predicted_rating"] = personalized_predictions

comparison["error"] = (
    comparison["rating"] - comparison["predicted_rating"]
)

print(comparison.head(10))


         userId  movieId  rating  predicted_rating     error
5954604  122380    31445     2.5          2.192558  0.307442
1460784   22380    56775     4.0          3.299378  0.700622
6631832  104339      356     2.5          3.842271 -1.342271
5396407   64877     6874     4.0          4.472426 -0.472426
8150654   63164     2762     5.0          4.408532  0.591468
8920330   78757     4589     4.0          4.105568 -0.105568
8950743   79382    72733     4.0          3.743684  0.256316
6838436   35374     7090     0.5          2.728680 -2.228680
4806124   88427     1196     5.0          4.825864  0.174136
2062392  134456     2355     4.0          3.759527  0.240473


## 5. Improving Predictions with Bayesian Smoothing

### Investigating Rating Frequency

Movies with very few ratings may have average ratings that do not reliably represent broader user preferences.

We examine the distribution of rating counts across movies to understand this problem.

The results show that many movies have relatively few ratings, with a median of four ratings per movie.

To address this, we apply Bayesian smoothing, which adjusts movie averages toward the global average based on the number of ratings received.

Movies with fewer ratings receive stronger adjustments, while movies with many ratings retain averages closer to their observed values.

We initially use a smoothing strength of 20.

The regularized personalized model achieves a validation RMSE of approximately 0.8867, improving upon the previous personalized model.

In [25]:

# Count the number of ratings received by each movie
movie_counts = train_set.groupby("movieId")["rating"].count()

# Display summary statistics
print(movie_counts.describe())


count    45438.000000
mean       176.064748
std        899.654389
min          1.000000
25%          1.000000
50%          4.000000
75%         24.000000
max      26242.000000
Name: rating, dtype: float64


In [26]:

# Set smoothing strength
m = 20

# Calculate regularized movie averages
regularized_movie_mean = (
    (movie_counts * movie_mean + m * global_mean)
    / (movie_counts + m)
)

# Display the first five results
print(regularized_movie_mean.head())


movieId
1    3.891517
2    3.264221
3    3.135836
4    2.902182
5    3.055924
Name: rating, dtype: float64


In [27]:

# Get regularized movie averages for validation data
val_regularized_movie = val_set["movieId"].map(
    regularized_movie_mean
)

# Handle movies missing from the training set
val_regularized_movie = val_regularized_movie.fillna(global_mean)

# Combine user averages with regularized movie averages
regularized_predictions = (
    val_user_mean + val_regularized_movie - global_mean
)

# Keep predictions within the valid rating range
regularized_predictions = regularized_predictions.clip(0.5, 5.0)

# Calculate RMSE
regularized_rmse = np.sqrt(
    mean_squared_error(actual_ratings, regularized_predictions)
)

# Compare model performance
print("Baseline RMSE:", rmse)
print("Personalized RMSE:", personalized_rmse)
print("Regularized RMSE:", regularized_rmse)


Baseline RMSE: 0.9620988813031842
Personalized RMSE: 0.88970814883535
Regularized RMSE: 0.886673861650974


### Tuning the Bayesian Smoothing Parameter

The smoothing parameter, m, controls how strongly movie averages are adjusted toward the global average.

We evaluate five values: 5, 10, 20, 50, and 100.

For each value, we calculate regularized movie averages, generate validation predictions, and measure RMSE.

The results show that m = 20 produces the lowest validation RMSE of approximately 0.886674.

Therefore, we select m = 20 for the final regularized prediction model.

This experiment demonstrates how hyperparameter tuning can improve prediction performance.

In [28]:

# Try different regularization strengths
m_values = [5, 10, 20, 50, 100]

results = []

for m in m_values:
    # Calculate smoothed movie averages
    smoothed_mean = (
        (movie_counts * movie_mean + m * global_mean)
        / (movie_counts + m)
    )

    # Predict validation ratings
    smoothed_movie = val_set["movieId"].map(smoothed_mean)
    smoothed_movie = smoothed_movie.fillna(global_mean)

    predictions = (
        val_user_mean + smoothed_movie - global_mean
    ).clip(0.5, 5.0)

    # Calculate RMSE
    score = np.sqrt(
        mean_squared_error(actual_ratings, predictions)
    )

    results.append((m, score))

# Display results
for m, score in results:
    print(f"m = {m}: RMSE = {score:.6f}")


m = 5: RMSE = 0.887204
m = 10: RMSE = 0.886876
m = 20: RMSE = 0.886674
m = 50: RMSE = 0.886807
m = 100: RMSE = 0.887579


### Examining User Rating Activity

We examine how frequently users rate movies to understand the distribution of user activity.

The results show that users contribute different numbers of ratings, with a median of 23 ratings per user.

Users with limited rating histories may have less reliable average ratings than users who have rated many movies.

This observation suggests an opportunity for further improvement through user-level regularization or more advanced collaborative filtering techniques.

In [29]:

# Count how many ratings each user has given
user_counts = train_set.groupby("userId")["rating"].count()

# Display summary statistics
print(user_counts.describe())


count    162538.000000
mean         49.219444
std          85.939784
min           1.000000
25%          11.000000
50%          23.000000
75%          52.000000
max       10353.000000
Name: rating, dtype: float64


## 6. Final Model and Kaggle Submission

### Preparing the Final Prediction Model

After evaluating the candidate models, we select the regularized personalized model with a Bayesian smoothing parameter of 20.

We recalculate the global average, user averages, movie averages, and movie rating counts using the complete training dataset.

Using all available training ratings allows the final model to incorporate more information before generating predictions for the Kaggle test dataset.

The regularized movie averages are then used alongside user averages to estimate unknown ratings.

In [30]:

# Prepare the final model using all training data

final_global_mean = train_data["rating"].mean()

final_user_mean = train_data.groupby("userId")["rating"].mean()

final_movie_mean = train_data.groupby("movieId")["rating"].mean()

final_movie_counts = train_data.groupby("movieId")["rating"].count()

best_m = 20

final_regularized_movie_mean = (
    (final_movie_counts * final_movie_mean + best_m * final_global_mean)
    / (final_movie_counts + best_m)
)

print("Final model prepared successfully!")


Final model prepared successfully!


### Generating Final Test Predictions

We use the final model to predict ratings for every user–movie pair in the Kaggle test dataset.

For users or movies not found in the training data, we substitute the global average rating.

We combine user averages and regularized movie averages, then restrict predictions to the valid rating range of 0.5 to 5.0.

Finally, we verify that the number of predictions matches the number of test records and that no missing values remain.

The model successfully generates 5,000,019 predictions, with values ranging from 0.5 to 5.0.

In [31]:

# Get user averages for the test dataset
test_user_mean = test_data["userId"].map(final_user_mean)

# Get regularized movie averages for the test dataset
test_movie_mean = test_data["movieId"].map(final_regularized_movie_mean)

# Handle missing users or movies
test_user_mean = test_user_mean.fillna(final_global_mean)
test_movie_mean = test_movie_mean.fillna(final_global_mean)

# Generate final predictions
final_predictions = (
    test_user_mean + test_movie_mean - final_global_mean
)

# Keep predictions within the rating range
final_predictions = final_predictions.clip(0.5, 5.0)

print(final_predictions.head())


0    3.926236
1    4.412378
2    3.923445
3    4.163500
4    3.492272
dtype: float64


In [32]:

# Check the final predictions

print("Total test records:", len(test_data))
print("Total predictions:", len(final_predictions))
print("Missing predictions:", final_predictions.isna().sum())
print("Minimum rating:", final_predictions.min())
print("Maximum rating:", final_predictions.max())


Total test records: 5000019
Total predictions: 5000019
Missing predictions: 0
Minimum rating: 0.5
Maximum rating: 5.0


### Creating the Kaggle Submission File

We prepare the final predictions in the format required by the ALX Movie Recommendation competition.

Each submission record contains:
- Id: A combination of userId and movieId separated by an underscore.
- rating: The predicted rating generated by the final model.

We save the results as `movie_submission.csv`, containing 5,000,019 predictions.

The submission file is then ready for evaluation on the Kaggle leaderboard.

In [37]:

# Create the Id column required by Kaggle
submission = pd.DataFrame()

submission["Id"] = (
    test_data["userId"].astype(str)
    + "_"
    + test_data["movieId"].astype(str)
)

# Add our final model predictions
submission["rating"] = final_predictions.to_numpy()

# Save the corrected submission
submission.to_csv("movie_submission.csv", index=False)

print("Kaggle submission created successfully!")
print(submission.head())
print("Submission shape:", submission.shape)


Kaggle submission created successfully!
       Id    rating
0  1_2011  3.926236
1  1_4144  4.412378
2  1_5767  3.923445
3  1_6711  4.163500
4  1_7318  3.492272
Submission shape: (5000019, 2)


## 7. Conclusion and Future Improvements

This project developed a movie recommendation and rating prediction system using the ALX Movie Recommendation dataset.

An item-based collaborative filtering approach was implemented using cosine similarity to identify movies with similar user rating patterns.

For rating prediction, three approaches were evaluated:

| Model | Validation RMSE |
|---|---:|
| Baseline movie-average model | 0.9621 |
| Personalized rating prediction model | 0.8897 |
| Bayesian-smoothed personalized model | 0.8867 |

The Bayesian-smoothed model achieved the lowest validation RMSE among the approaches tested, demonstrating the benefit of accounting for both individual user rating behaviour and the reliability of movie averages.

The final model generated predictions for 5,000,019 test records without missing values.

### Future Improvements

Potential improvements include:

- Applying regularization to user averages.
- Exploring matrix factorization techniques such as Singular Value Decomposition (SVD).
- Improving the recommendation function to display movie titles and similarity scores.
- Evaluating recommendation quality using ranking-based metrics.
- Comparing additional models using consistent validation procedures.

These improvements could help produce more accurate rating predictions and more useful personalized movie recommendations.